In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

eth = pd.read_csv("../data/ethiopia_clean.csv", index_col="Date", parse_dates=True)
ken = pd.read_csv("../data/kenya_clean.csv", index_col="Date", parse_dates=True)
nig = pd.read_csv("../data/nigeria_clean.csv", index_col="Date", parse_dates=True)
sud = pd.read_csv("../data/sudan_clean.csv", index_col="Date", parse_dates=True)
tan = pd.read_csv("../data/tanzania_clean.csv", index_col="Date", parse_dates=True)

all_countries = pd.concat([eth, ken, nig, sud, tan])
all_countries = all_countries.sort_index()

In [ ]:
baseline = all_countries.loc['2015': '2019']
current = all_countries.loc['2020': '2026']

baseline_avg = baseline.groupby('Country')['T2M'].mean()
current_avg = current.groupby('Country')['T2M'].mean()

anomaly = current_avg - baseline_avg
print("Temperature Anomaly (Current vs Baseline):", end=" ")
print(anomaly.sort_values(ascending=False))

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=all_countries, x='Country', y='T2M', palette='Set2')
plt.title('Temperature Variability Across Target Nations (2015-2026)')
plt.ylabel('Temperature (°C)')
plt.show()

In [ ]:
precip_volatility = all_countries.groupby('Country')['PRECTOTCORR'].std()

ranking_df = pd.DataFrame({
    'Temp_Anomaly': anomaly,
    'Precip_Volatility': precip_volatility
})

ranking_df['Vulnerability_Score'] = ranking_df['Temp_Anomaly'] + (ranking_df['Precip_Volatility'] / 10)
print(ranking_df.sort_values(by='Vulnerability_Score', ascending=False))

In [ ]:
all_countries['Heatwave'] = all_countries['T2M_MAX'] > 35
heat_counts = all_countries.groupby(['Country', all_countries.index.year])['Heatwave'].sum().reset_index()

plt.figure(figsize=(12, 6))
sns.barplot(data=heat_counts, x='Date', y='Heatwave', hue='Country')
plt.title('Annual Count of Extreme Heat Days (>35°C)')
plt.ylabel('Number of Days')
plt.show()

def max_consecutive_dry(series):
    is_dry = series < 1
    return is_dry.groupby((~is_dry).cumsum()).sum().max()

drought_stats = all_countries.groupby(['Country', all_countries.index.year])['PRECTOTCORR'].apply(max_consecutive_dry)